# Chest X-Ray Classification — Pneumonia Detection

**Patient Health Monitoring using Chest X-Ray Classification**

ResNet50 transfer learning for binary classification (NORMAL vs PNEUMONIA).

---
### Week 2 — Computer Vision Term Project

## 1. Setup

In [ ]:
print("hello world")

In [ ]:
!pip install -q torch torchvision matplotlib scikit-learn pillow tqdm requests 2>&1 | tail -5

In [ ]:
import os, sys, copy, shutil, random, json
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision import datasets, transforms, models

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)
from PIL import Image
from tqdm import tqdm
import requests
import zipfile

print(f"PyTorch {torch.__version__}, GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

## 2. Download Dataset

In [ ]:
DATA_URL = "https://data.mendeley.com/public-files/datasets/rscbjbr9sj/files/f12eaf6d-6023-432f-acc9-80c9d7393433/file_downloaded"
ZIP_PATH = "ChestXRay2017.zip"
DATA_DIR = Path("chest_xray")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/125.0.0.0 Safari/537.36",
    "Accept": "application/zip,application/octet-stream,*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://data.mendeley.com/datasets/rscbjbr9sj/2",
}

if not DATA_DIR.exists():
    print("Downloading dataset (1.15 GB)...")
    resp = requests.get(DATA_URL, headers=HEADERS, stream=True, timeout=300)
    resp.raise_for_status()
    total = int(resp.headers.get("content-length", 0))
    with open(ZIP_PATH, "wb") as f:
        with tqdm(total=total, unit="B", unit_scale=True, desc="chest_xray.zip") as pbar:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))
    print("\nExtracting...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall()
    print("Done.")
else:
    print(f"{DATA_DIR}/ already exists — skipping download.")

In [ ]:
splits = ["train", "test"]
classes = ["NORMAL", "PNEUMONIA"]
for s in splits:
    for c in classes:
        d = DATA_DIR / s / c
        count = len(list(d.glob("*"))) if d.exists() else 0
        print(f"{s}/{c}: {count} images")

assert DATA_DIR.exists(), "Dataset directory not found"

## 3. Dataset Cleaning

Removes corrupt images, deduplicates by MD5, resizes to 224×224 RGB.

In [ ]:
# Run the clean_dataset.py from the repo
if not DATA_DIR.exists():
    raise RuntimeError("Download the dataset first.")

!python clean_dataset.py

## 4. Stratified Train / Validation Split

20% of training data held out for validation (stratified by class).

In [ ]:
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
VAL_SPLIT = 0.2
BATCH_SIZE = 32

# Collect all file paths with their class labels
train_files, train_labels = [], []
for cls in classes:
    cls_dir = TRAIN_DIR / cls
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith((".jpeg", ".jpg", ".png")):
            train_files.append(str(cls_dir / fname))
            train_labels.append(cls)

# Stratified split (maintains class proportion in val)
trn_idx, val_idx = train_test_split(
    np.arange(len(train_files)),
    test_size=VAL_SPLIT,
    stratify=train_labels,
    random_state=42,
)

print(f"Train: {len(trn_idx)}  Val: {len(val_idx)}  Test: {len(list(TEST_DIR.rglob('*')))}")

# Count class distribution in each split
for name, idx in [("Train", trn_idx), ("Val", val_idx)]:
    labels = [train_labels[i] for i in idx]
    norm = labels.count("NORMAL")
    pneu = labels.count("PNEUMONIA")
    print(f"{name:>6}: NORMAL={norm:>4}  PNEUMONIA={pneu:>4}  ({norm/pneu:.2f}:1 ratio)")

## 5. Data Transforms & Loaders

In [ ]:
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

full_train_dataset = datasets.ImageFolder(str(TRAIN_DIR), transform=train_transform)
val_dataset = datasets.ImageFolder(str(TRAIN_DIR), transform=eval_transform)
test_dataset = datasets.ImageFolder(str(TEST_DIR), transform=eval_transform)

class_names = full_train_dataset.classes
print(f"Classes: {class_names}")

train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE,
                          sampler=SubsetRandomSampler(trn_idx),
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        sampler=SubsetRandomSampler(val_idx),
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# Show a few training samples
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
data_iter = iter(train_loader)
images, labels = next(data_iter)

mean = torch.tensor(IMG_MEAN).view(3, 1, 1)
std = torch.tensor(IMG_STD).view(3, 1, 1)

for i in range(8):
    ax = axes[i // 4][i % 4]
    img = images[i] * std + mean
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(class_names[labels[i]])
    ax.axis("off")
plt.tight_layout()
plt.savefig("week2_classification/sample_batch.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Model — ResNet50 with Transfer Learning

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}  Trainable: {trainable_params:,}")

In [ ]:
# Weighted loss to handle imbalance
label_counts = [train_labels.count(c) for c in class_names]
total_count = sum(label_counts)
class_weights = [total_count / c for c in label_counts]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

for c, w in zip(class_names, class_weights):
    print(f"  {c:>10}: weight = {w:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

## 7. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss, running_correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc="Train", leave=False)

    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()

        with torch.amp.autocast(device_type=device.type):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=loss.item())

    return running_loss / total, running_correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, running_correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Eval", leave=False):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type):
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            running_correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, running_correct / total, all_preds, all_labels

In [ ]:
OUTPUT_DIR = Path("week2_classification")
OUTPUT_DIR.mkdir(exist_ok=True)

EPOCHS_HEAD = 20
EPOCHS_FULL = 15
LR_HEAD = 1e-3
LR_FULL = 1e-4
EARLY_STOP_PATIENCE = 5

best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

scaler = torch.amp.GradScaler(device=device.type)

### Phase 1: Train Head Only (backbone frozen)

In [ ]:
# Freeze backbone
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.fc.parameters(), lr=LR_HEAD)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  patience=3, factor=0.1, verbose=True)

print("=" * 55)
print("Phase 1 — Training classifier head (backbone frozen)")
print("=" * 55)

no_improve_epochs = 0
for epoch in range(1, EPOCHS_HEAD + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:2d}/{EPOCHS_HEAD}  "
          f"Train loss: {train_loss:.4f}  acc: {train_acc:.4f}  "
          f"Val loss: {val_loss:.4f}  acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        no_improve_epochs = 0
        torch.save(model.state_dict(), OUTPUT_DIR / "best_model_head.pth")
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= EARLY_STOP_PATIENCE:
            print(f"Early stopping triggered after {epoch} epochs.")
            break

print(f"\nBest val acc (Phase 1): {best_val_acc:.4f}")

### Phase 2: Fine-Tune Full Model

In [ ]:
# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=LR_FULL)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  patience=3, factor=0.1, verbose=True)

print("=" * 55)
print("Phase 2 — Fine-tuning all layers")
print("=" * 55)

no_improve_epochs = 0
for epoch in range(1, EPOCHS_FULL + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:2d}/{EPOCHS_FULL}  "
          f"Train loss: {train_loss:.4f}  acc: {train_acc:.4f}  "
          f"Val loss: {val_loss:.4f}  acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        no_improve_epochs = 0
        torch.save(model.state_dict(), OUTPUT_DIR / "best_model.pth")
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= EARLY_STOP_PATIENCE:
            print(f"Early stopping triggered after {epoch} epochs.")
            break

print(f"\nBest val acc (overall): {best_val_acc:.4f}")

In [ ]:
# Load best model
model.load_state_dict(best_model_wts)

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs, history["train_loss"], label="Train", marker="o", markersize=3)
axes[0].plot(epochs, history["val_loss"], label="Val", marker="s", markersize=3)
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mark phase boundary
if len(epochs) > EPOCHS_HEAD:
    axes[0].axvline(x=EPOCHS_HEAD + 0.5, color='gray', linestyle='--', alpha=0.5)

axes[1].plot(epochs, history["train_acc"], label="Train", marker="o", markersize=3)
axes[1].plot(epochs, history["val_acc"], label="Val", marker="s", markersize=3)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

if len(epochs) > EPOCHS_HEAD:
    axes[1].axvline(x=EPOCHS_HEAD + 0.5, color='gray', linestyle='--', alpha=0.5,
                    label="Phase boundary")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR / 'training_curves.png'}")

## 9. Evaluation on Held-Out Test Set

In [ ]:
test_loss, test_acc, y_pred, y_true = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}  Test Accuracy: {test_acc:.4f}\n")

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=[0, 1], average=None
)

print(f"{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("-" * 44)
for i, c in enumerate(class_names):
    print(f"{c:<12} {precision[i]:>10.4f} {recall[i]:>10.4f} {f1[i]:>10.4f}")
print("-" * 44)

macro_f1 = np.mean(f1)
weighted_prec, weighted_rec, weighted_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='weighted'
)
print(f"{'Macro F1':<12} {macro_f1:>10.4f}")
print(f"{'Weighted F1':<12} {weighted_f1:>10.4f}")
print()
print(f"Misclassifications: {int((1 - test_acc) * len(y_true))} / {len(y_true)}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ROC Curve
model.eval()
all_probs = []
with torch.no_grad():
    for images, _ in tqdm(test_loader, desc="Computing probs"):
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type):
            outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs[:, 1].cpu())

y_prob = torch.cat(all_probs).numpy()

fpr, tpr, thresholds = roc_curve(y_true, y_prob, pos_label=1)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — PNEUMONIA Class')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"AUC: {roc_auc:.4f}")

In [ ]:
# Generate sample_predictions.png — correct and wrong predictions on test images
model.eval()
num_samples = 16
cols, rows = 4, 4
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))

collected = []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type):
            outputs = model(images)
        preds = outputs.argmax(1)
        for i in range(images.size(0)):
            collected.append((images[i].cpu(), labels[i].item(), preds[i].item()))
            if len(collected) >= num_samples:
                break
        if len(collected) >= num_samples:
            break

mean = torch.tensor(IMG_MEAN).view(3, 1, 1)
std = torch.tensor(IMG_STD).view(3, 1, 1)

for i, (img, true_label, pred_label) in enumerate(collected):
    ax = axes[i // cols][i % cols]
    img_vis = img * std + mean
    img_vis = img_vis.clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img_vis)
    correct = true_label == pred_label
    color = "green" if correct else "red"
    title = f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}"
    ax.set_title(title, fontsize=9, color=color)
    ax.axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {OUTPUT_DIR / 'sample_predictions.png'}")

## 10. Save Report & Metrics

In [ ]:
lines = []
lines.append("=" * 60)
lines.append("Chest X-Ray Classification — Test Set Evaluation Report")
lines.append("=" * 60)
lines.append(f"Model: ResNet50 (transfer learning, ImageNet weights)")
lines.append(f"Dataset: Chest X-Ray (Pneumonia) — {len(y_true)} test images")
lines.append(f"Input size: 224x224 RGB")
lines.append("")
lines.append(f"Test Loss:       {test_loss:.4f}")
lines.append(f"Test Accuracy:   {test_acc:.4f}")
lines.append(f"Macro F1:        {macro_f1:.4f}")
lines.append(f"Weighted F1:     {weighted_f1:.4f}")
lines.append(f"ROC AUC:         {roc_auc:.4f}")
lines.append("")
lines.append(f"{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
lines.append("-" * 44)
for i, c in enumerate(class_names):
    lines.append(f"{c:<12} {precision[i]:>10.4f} {recall[i]:>10.4f} {f1[i]:>10.4f}")
lines.append("-" * 44)
lines.append("")
lines.append(f"Confusion Matrix:")
lines.append(str(cm))
lines.append("")
lines.append(f"Misclassifications: {int((1 - test_acc) * len(y_true))} / {len(y_true)}")

report_path = OUTPUT_DIR / "classification_report.txt"
report_path.write_text("\n".join(lines))
print(f"Report saved to {report_path}")

# Export metrics.json for programmatic consumption
metrics = {
    "accuracy": round(test_acc, 4),
    "loss": round(test_loss, 4),
    "macro_f1": round(macro_f1, 4),
    "weighted_f1": round(weighted_f1, 4),
    "roc_auc": round(roc_auc, 4),
    "precision": {class_names[i]: round(float(precision[i]), 4) for i in range(len(class_names))},
    "recall": {class_names[i]: round(float(recall[i]), 4) for i in range(len(class_names))},
    "f1": {class_names[i]: round(float(f1[i]), 4) for i in range(len(class_names))},
    "confusion_matrix": cm.tolist(),
    "num_test_samples": len(y_true),
    "misclassifications": int((1 - test_acc) * len(y_true)),
}
metrics_path = OUTPUT_DIR / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Metrics saved to {metrics_path}")
print("\n".join(lines))

## 11. Summary

All outputs saved to `week2_classification/`:

| File | Description |
|------|-------------|
| `best_model.pth` | Best model checkpoint |
| `training_curves.png` | Loss & accuracy over epochs |
| `confusion_matrix.png` | Test set confusion matrix |
| `roc_curve.png` | ROC curve with AUC |
| `sample_predictions.png` | Grid of correct (green) / wrong (red) predictions |
| `metrics.json` | Machine-readable accuracy, precision, recall, f1, confusion matrix |
| `classification_report.txt` | Full text report |

In [ ]:
!zip -r data.zip week2_classification

In [ ]:
from google.colab import files
files.download('data.zip')